# YOLOv8s 通用目标检测训练 · Colab

先在「运行时 → 更改运行时类型」中选择 GPU，再依次执行下方单元格。

- 目标：训练人员、车辆、动物等通用目标检测器，随后逐帧应用于运动目标视频；本 Notebook 不训练动作识别或目标跟踪。
- 数据：暂按 COCO 2017 的 80 类设置，训练 118,287 张、验证 5,000 张；首次训练会自动下载完整训练与验证数据，需预留下载、解压空间和训练时间。不是 COCO8/COCO128 演示子集。
- 权重：yolov8s.pt（已在 COCO 上预训练）。普通视频检测可直接用这份权重；再次在 COCO 上训练属于继续训练，不保证优于原权重。
- 参数：100 epochs、imgsz=640、batch=32；显存不足时将 batch 改为 16 或 8。
- 输出：MyDrive/YOLO_runs/yolov8s_coco_<北京时间>/，包含 weights/best.pt、weights/last.pt、results.csv、args.yaml 和图表。
- 训练、权重检查及 Drive 同步成功后自动释放运行时；出错时保留连接。同步耗时取决于 Drive，不再限制为 60 秒。
- 安装与训练使用命令行，避免 Notebook 内核之前缓存的 ultralytics 同名目录影响训练。
- 若当前 Colab 已有旧 Notebook，请替换单元格或从 GitHub 重新打开；旧页面不会自动更新。

参考：[Ultralytics CLI](https://docs.ultralytics.com/usage/cli/)、[COCO 数据集](https://docs.ultralytics.com/datasets/detect/coco/)。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%cd /content
!test -d /content/YOLOv8_jetson/.git || git clone --recurse-submodules https://github.com/o00ney/YOLOv8_jetson.git
%cd /content/YOLOv8_jetson
!git submodule update --init --recursive
%cd /content/YOLOv8_jetson/ultralytics
%pip install -e .
!nvidia-smi
!yolo checks
!yolo settings datasets_dir=/content/datasets wandb=False comet=False mlflow=False


In [ ]:
import subprocess
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import drive, runtime

# 使用北京时间为每次实验命名
run_name = datetime.now(ZoneInfo("Asia/Shanghai")).strftime(
    "yolov8s_coco_%Y%m%d_%H%M%S"
)
project = "/content/drive/MyDrive/YOLO_runs"

# 确认 Drive 已挂载，避免将结果误写到临时磁盘
subprocess.run(["test", "-d", "/content/drive/MyDrive"], check=True)

# COCO 会按官方配置自动下载到 /content/datasets
subprocess.run(
    [
        "yolo", "detect", "train",
        "model=yolov8s.pt",
        "data=coco.yaml",
        "epochs=100",
        "imgsz=640",
        "batch=32",
        "device=0",
        "workers=2",
        "save=True",
        "plots=True",
        f"project={project}",
        f"name={run_name}",
    ],
    check=True,
)

# 确认权重已生成；训练或检查失败时保留运行时
for weight in ("best.pt", "last.pt"):
    subprocess.run(
        ["test", "-s", f"{project}/{run_name}/weights/{weight}"],
        check=True,
    )

# 将云盘待写入内容同步完成，再释放运行时
drive.flush_and_unmount()
runtime.unassign()
